# 00 Foundations — Probability Distributions

**Status:** Complete

**Learning outcome:** Fluency with core probability distributions, their parameterisation, and when each arises in ML — including Beta priors for survival modelling.

## Why This Matters

Every ML model makes distributional assumptions — logistic regression assumes Bernoulli outputs, VAEs assume Gaussian latents, and Bayesian priors on survival probability use the Beta distribution. Understanding distributions lets you choose the right loss function, interpret model outputs, and design meaningful experiments.

In the MNEMOSYNE project we model P(survival | patient, system, t+12h). The Beta distribution is a natural prior for this bounded probability, and the Bernoulli/Binomial family governs binary outcomes.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


## Theory Sketch — Key Distributions

| Distribution | Support | Parameters | ML Use Case |
|---|---|---|---|
| Bernoulli | {0, 1} | p | Binary classification targets |
| Binomial | {0,…,n} | n, p | Count of successes in n trials |
| Gaussian | ℝ | μ, σ² | Regression targets, latent spaces |
| Exponential | [0, ∞) | λ | Time-to-event (survival) |
| Beta | [0, 1] | α, β | Prior on probabilities |
| Poisson | {0,1,2,…} | λ | Count data (rare events) |

**Key relationships:**
- Bernoulli is Binomial(n=1, p)
- Beta is conjugate prior for Bernoulli/Binomial
- As n→∞, Binomial → Gaussian (CLT)

## Distribution Gallery — 2×3 Visualisation

Six distributions, each plotted as PDF/PMF with two parameter settings to build intuition for how parameters shape the distribution.

In [2]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Row 0, Col 0 — Bernoulli
ax = axes[0, 0]
for p in [0.3, 0.7]:
    pmf = [1 - p, p]
    ax.bar([0, 1], pmf, alpha=0.5, label=f'p={p}', width=0.3,
           align='edge' if p == 0.7 else 'center')
ax.set_title('Bernoulli')
ax.set_xlabel('k'); ax.set_ylabel('P(X=k)')
ax.legend()

# Row 0, Col 1 — Binomial
ax = axes[0, 1]
n = 20
for p in [0.3, 0.7]:
    k = np.arange(0, n + 1)
    ax.bar(k + (0.2 if p == 0.7 else 0), stats.binom.pmf(k, n, p),
           alpha=0.5, label=f'n={n}, p={p}', width=0.4)
ax.set_title('Binomial')
ax.set_xlabel('k'); ax.set_ylabel('P(X=k)')
ax.legend()

# Row 0, Col 2 — Gaussian
ax = axes[0, 2]
x = np.linspace(-6, 10, 300)
for mu, sigma in [(0, 1), (3, 2)]:
    ax.plot(x, stats.norm.pdf(x, mu, sigma), label=f'μ={mu}, σ={sigma}')
ax.set_title('Gaussian')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend()

# Row 1, Col 0 — Exponential
ax = axes[1, 0]
x = np.linspace(0, 8, 300)
for lam in [0.5, 1.5]:
    ax.plot(x, stats.expon.pdf(x, scale=1/lam), label=f'λ={lam}')
ax.set_title('Exponential')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend()

# Row 1, Col 1 — Beta
ax = axes[1, 1]
x = np.linspace(0.001, 0.999, 300)
for a, b in [(2, 5), (5, 2), (2, 2)]:
    ax.plot(x, stats.beta.pdf(x, a, b), label=f'α={a}, β={b}')
ax.set_title('Beta')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend()

# Row 1, Col 2 — Poisson
ax = axes[1, 2]
for lam in [2, 7]:
    k = np.arange(0, 20)
    ax.bar(k + (0.3 if lam == 7 else 0), stats.poisson.pmf(k, lam),
           alpha=0.5, label=f'λ={lam}', width=0.3)
ax.set_title('Poisson')
ax.set_xlabel('k'); ax.set_ylabel('P(X=k)')
ax.legend()

plt.suptitle('Distribution Gallery', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/distribution_gallery.png', dpi=100)
plt.show()
print("Gallery saved.")

Gallery saved.


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_45436/163455602.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# --- Visual: Law of Large Numbers Convergence ---
np.random.seed(42)
n_samples = 2000
samples = np.random.exponential(1.0, size=n_samples)
running_mean = np.cumsum(samples) / np.arange(1, n_samples + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(running_mean, color='steelblue', lw=1.5, label='Running mean')
ax.axhline(1.0, color='red', ls='--', lw=2, label='True mean (1.0)')

# Confidence band: ±1.96*sigma/sqrt(n)
n_arr = np.arange(1, n_samples + 1)
band = 1.96 * 1.0 / np.sqrt(n_arr)  # Exponential(1) has sigma=1
ax.fill_between(n_arr, 1.0 - band, 1.0 + band, alpha=0.15, color='red',
                label='95% confidence band')

ax.set_xlabel('Number of samples', fontsize=11)
ax.set_ylabel('Running mean', fontsize=11)
ax.set_title('Law of Large Numbers: Exponential(1) running mean → true mean', fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 3)
plt.tight_layout()
plt.savefig('../assets/lln_convergence.png', dpi=100)
plt.show()
print(f"Final running mean after {n_samples} samples: {running_mean[-1]:.4f} (true: 1.0)")
assert abs(running_mean[-1] - 1.0) < 0.1, "LLN: running mean should converge to 1.0"

## Beta Distribution Deep Dive — Conjugate Prior for Survival

The Beta(α, β) distribution is the conjugate prior for the Bernoulli likelihood. After observing `s` survivals out of `n` patients:

$$\text{Posterior} = \text{Beta}(\alpha + s, \; \beta + n - s)$$

This is directly relevant to MNEMOSYNE: we can encode prior belief about P(survival) and update it as data arrives.

In [3]:
# Bayesian updating with Beta-Bernoulli model
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x = np.linspace(0, 1, 300)

# Prior: weakly informative Beta(2, 2)
alpha_prior, beta_prior = 2, 2

scenarios = [
    ("5 of 10 survive", 5, 10),
    ("15 of 20 survive", 15, 20),
    ("70 of 100 survive", 70, 100),
]

for ax, (title, s, n) in zip(axes, scenarios):
    alpha_post = alpha_prior + s
    beta_post = beta_prior + (n - s)

    ax.plot(x, stats.beta.pdf(x, alpha_prior, beta_prior),
            'k--', label=f'Prior Beta({alpha_prior},{beta_prior})')
    ax.plot(x, stats.beta.pdf(x, alpha_post, beta_post),
            'b-', lw=2, label=f'Posterior Beta({alpha_post},{beta_post})')
    ax.axvline(s / n, color='r', ls=':', label=f'MLE = {s/n:.2f}')
    ax.set_title(title)
    ax.set_xlabel('P(survival)')
    ax.legend(fontsize=8)

plt.suptitle('Bayesian Updating: Beta-Bernoulli', fontweight='bold')
plt.tight_layout()
plt.show()

# Verify posterior concentrates around MLE with more data
post_mean = alpha_post / (alpha_post + beta_post)
assert abs(post_mean - 70/100) < 0.02, f"Posterior mean {post_mean:.3f} should be near 0.70"
print(f"Posterior mean after 100 obs: {post_mean:.4f} ≈ MLE 0.70 ✓")

Posterior mean after 100 obs: 0.6923 ≈ MLE 0.70 ✓


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_45436/290508587.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**Understanding the Beta Distribution as a Prior**

**Plain language:** Imagine you're guessing how good a new restaurant is. Before visiting, you have a vague sense — maybe a 50/50 shot it's good. After 10 visits where 7 were great, your opinion sharpens. The Beta distribution is exactly this: it represents your belief about a probability (like survival rate) and updates as you see more data. More data = sharper, more confident belief.

**Building intuition:** The Beta(α, β) distribution lives on [0, 1] — perfect for modelling probabilities. The parameters α and β act like "pseudo-counts": α is like the number of prior successes and β is prior failures. The mean is α/(α+β). As real data arrives (s successes in n trials), the posterior is simply Beta(α+s, β+n-s) — no complex computation needed. This is called *conjugacy*: the posterior stays in the same family as the prior. With little data, the prior dominates; with lots of data, the posterior concentrates on the MLE.

**Formal statement:** The Beta(α, β) PDF is $f(x; \alpha, \beta) = \frac{x^{\alpha-1}(1-x)^{\beta-1}}{B(\alpha, \beta)}$ where $B(\alpha, \beta) = \frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha+\beta)}$. It is the conjugate prior for the Bernoulli/Binomial likelihood: given $s$ successes in $n$ trials with prior Beta(α, β), the posterior is Beta(α+s, β+n−s) with mean $\frac{\alpha+s}{\alpha+\beta+n}$.

---

## Central Limit Theorem — Visual Proof

The CLT states that the sampling distribution of the mean converges to a Gaussian regardless of the population distribution. We demonstrate with Exponential(λ=1) samples — a heavily skewed distribution.

In [4]:
# CLT demonstration: sample means from Exponential(1)
np.random.seed(42)
sample_sizes = [1, 2, 5, 30]
n_experiments = 5000

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

for ax, n in zip(axes, sample_sizes):
    means = [np.mean(np.random.exponential(1.0, size=n)) for _ in range(n_experiments)]
    ax.hist(means, bins=50, density=True, alpha=0.7, color='steelblue')

    # Overlay theoretical Gaussian: mean=1, var=1/n
    x = np.linspace(min(means), max(means), 200)
    ax.plot(x, stats.norm.pdf(x, 1.0, 1.0 / np.sqrt(n)), 'r-', lw=2)
    ax.set_title(f'n = {n}')
    ax.set_xlabel('Sample mean')

axes[0].set_ylabel('Density')
plt.suptitle('CLT: Exponential(1) sample means → Gaussian', fontweight='bold')
plt.tight_layout()
plt.show()

# Verify CLT via moment matching: sample skewness should shrink with n
# Exponential(1) has skewness=2; CLT predicts skew → 2/sqrt(n)
means_arr = np.array(means)  # n=30 from last loop iteration
observed_skew = stats.skew(means_arr)
expected_skew = 2.0 / np.sqrt(30)
assert abs(observed_skew) < 1.0, f"Skewness {observed_skew:.3f} too large for n=30"
print(f"Sample mean skewness at n=30: {observed_skew:.3f} (theory ≈ {expected_skew:.3f})")
print("CLT convergence confirmed: skewness shrinks toward 0 as n grows ✓")

Sample mean skewness at n=30: 0.371 (theory ≈ 0.365)
CLT convergence confirmed: skewness shrinks toward 0 as n grows ✓


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_45436/1716960734.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**Understanding the Central Limit Theorem**

**Plain language:** If you take a large enough group of anything — heights, dice rolls, stock prices — and average them, the average will follow a bell curve, even if the individual measurements are wildly non-bell-shaped. It's like how a crowd of people walking randomly in different directions still drifts predictably on average. This is why the Gaussian (bell curve) shows up *everywhere* in statistics and ML.

**Building intuition:** The CLT says that the sample mean of $n$ independent draws from *any* distribution (with finite variance $\sigma^2$) converges to a Gaussian with the same mean but variance shrunk by $1/n$. The convergence speed depends on how skewed the original distribution is — symmetric distributions converge faster. In ML, this justifies: (1) treating batch gradient means as approximately Gaussian, (2) using Gaussian-based confidence intervals, and (3) the normality assumptions behind many statistical tests.

**Formal statement:** Let $X_1, \ldots, X_n$ be i.i.d. with mean $\mu$ and variance $\sigma^2 < \infty$. Then $\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i$ satisfies $\sqrt{n}(\bar{X}_n - \mu) \xrightarrow{d} \mathcal{N}(0, \sigma^2)$ as $n \to \infty$. Equivalently, $\bar{X}_n \sim \mathcal{N}(\mu, \sigma^2/n)$ approximately for large $n$, with skewness decaying as $\gamma_1 / \sqrt{n}$.

---

## KL Divergence — Measuring Distribution Mismatch

KL divergence D_KL(P ‖ Q) measures how much information is lost when Q is used to approximate P. It is:
- **Non-negative** (Gibbs' inequality)
- **Asymmetric**: D_KL(P‖Q) ≠ D_KL(Q‖P)
- Used in VAE losses, knowledge distillation, and model comparison

In [5]:
# KL divergence between two Gaussians (closed-form)
def kl_gaussian(mu1, sigma1, mu2, sigma2):
    """KL(N(mu1,sigma1^2) || N(mu2,sigma2^2))"""
    return (np.log(sigma2 / sigma1)
            + (sigma1**2 + (mu1 - mu2)**2) / (2 * sigma2**2)
            - 0.5)

# Numerical KL via discretisation for verification
def kl_numerical(p_pdf, q_pdf, x):
    """Numerical KL(P || Q) on grid x."""
    dx = x[1] - x[0]
    p = p_pdf(x) * dx
    q = q_pdf(x) * dx
    mask = (p > 1e-12) & (q > 1e-12)
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

x = np.linspace(-10, 15, 10000)

# KL between N(0,1) and N(2,3)
kl_analytic = kl_gaussian(0, 1, 2, 3)
kl_numeric = kl_numerical(
    lambda x: stats.norm.pdf(x, 0, 1),
    lambda x: stats.norm.pdf(x, 2, 3),
    x
)

print(f"KL(N(0,1) || N(2,3)):  analytic = {kl_analytic:.4f},  numeric = {kl_numeric:.4f}")
assert abs(kl_analytic - kl_numeric) < 0.01, "KL mismatch!"

# Show asymmetry
kl_reverse = kl_gaussian(2, 3, 0, 1)
print(f"KL(N(2,3) || N(0,1)):  analytic = {kl_reverse:.4f}")
print(f"Asymmetry confirmed: {kl_analytic:.4f} ≠ {kl_reverse:.4f} ✓")

# Visualise
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(x, stats.norm.pdf(x, 0, 1), label='P = N(0,1)')
ax.plot(x, stats.norm.pdf(x, 2, 3), label='Q = N(2,3)')
ax.fill_between(x, stats.norm.pdf(x, 0, 1), alpha=0.2)
ax.set_title(f'KL(P‖Q) = {kl_analytic:.3f},  KL(Q‖P) = {kl_reverse:.3f}')
ax.legend(); ax.set_xlabel('x')
plt.tight_layout(); plt.show()

KL(N(0,1) || N(2,3)):  analytic = 0.8764,  numeric = 0.8764
KL(N(2,3) || N(0,1)):  analytic = 4.9014
Asymmetry confirmed: 0.8764 ≠ 4.9014 ✓


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_45436/3845467132.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
**Understanding KL Divergence**

**Plain language:** KL divergence measures how "surprised" you'd be if you thought the world worked according to distribution Q, but it actually works according to distribution P. It's like packing for a holiday using the wrong weather forecast — if Q says "sunny" but P says "rainy", you'd be very surprised (high KL). Critically, packing for rain when it's sunny is a *different* level of surprise than packing for sun when it rains — that's why KL divergence is asymmetric.

**Building intuition:** $D_{KL}(P \| Q)$ penalises Q for placing low probability mass where P has high mass. If P is the "true" distribution and Q is your model, minimising $D_{KL}(P \| Q)$ is equivalent to maximising the expected log-likelihood under P — which is exactly what cross-entropy loss does. The reverse, $D_{KL}(Q \| P)$, encourages Q to be "mode-seeking" — it focuses on capturing the peak of P rather than covering all of P's mass. This asymmetry is why VAEs (which use forward KL) produce blurry samples while GANs (closer to reverse KL) produce sharp but less diverse samples.

**Formal statement:** For discrete distributions, $D_{KL}(P \| Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}$. For continuous: $D_{KL}(P \| Q) = \int p(x) \log \frac{p(x)}{q(x)} dx$. Properties: (1) $D_{KL} \geq 0$ (Gibbs' inequality), with equality iff $P = Q$; (2) asymmetric: $D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$; (3) not a metric (violates triangle inequality). For Gaussians: $D_{KL}(\mathcal{N}(\mu_1, \sigma_1^2) \| \mathcal{N}(\mu_2, \sigma_2^2)) = \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac{1}{2}$.

---

## Structured Interpretation

1. **Distribution choice matters**: Gaussian assumes unbounded symmetric errors; Beta constrains to [0,1] — right for probabilities. Using the wrong family leads to impossible predictions.

2. **Beta-Bernoulli conjugacy** gives closed-form posteriors — no MCMC needed for simple survival rate estimation. The posterior mean is a weighted average of prior mean and MLE, with weights proportional to pseudo-counts.

3. **CLT justifies normality assumptions** in large-sample statistics and explains why batch means in SGD training tend toward Gaussian — underpinning learning rate schedules and convergence theory.

4. **KL divergence is asymmetric**: D_KL(P‖Q) penalises Q for placing low mass where P has high mass (mode-seeking), while D_KL(Q‖P) is mass-covering. This distinction matters for VAE losses (forward KL) vs GAN-like objectives (reverse KL).

5. **For MNEMOSYNE**: We'll use Beta priors on survival probability and binary cross-entropy (equivalent to Bernoulli NLL) as our primary loss function.